# Label Similarity / Co-occurrence Analysis

1. Drops comments with any missing label value (the residual chunk 5212/5328 issue) from `score_sample_wide.parquet`.
2. Tags each of the 14 labels as `lloom` or `taxonomy`, using the exact mapping confirmed in `agreement_results.csv` (2.5.3's validation output) -- not reconstructed independently.
3. Saves the cleaned, per-comment table as `score_updated.jsonl`.
4. Computes **directional** co-occurrence: for label A, what % of the comments that matched A also matched label B (`P(B=1 | A=1)`) -- not the symmetric Jaccard from 2.3.4's redundancy check. This is asymmetric on purpose: "80% of Emotional Support also got esteem_support" is a one-way statement, and the reverse direction is usually a different number.
   - (a) within LLooM concepts only
   - (b) LLooM → taxonomy
   - (c) taxonomy → LLooM (the reverse direction -- genuinely different question, not redundant with (b))


## Setup

In [7]:
import os
import json

import pandas as pd
import numpy as np

pd.set_option('display.max_colwidth', 200)
pd.set_option('display.max_columns', 20)

OUTPUT_DIR = "/Users/nadia/Desktop/redditRun_june/comment_data/"
WIDE_PATH = os.path.join(OUTPUT_DIR, "score_sample_wide.parquet")
AGREEMENT_PATH = os.path.join(OUTPUT_DIR, "validation", "agreement_results.csv")
RESULTS_DIR = os.path.join(OUTPUT_DIR, "score_results_similarity")
os.makedirs(RESULTS_DIR, exist_ok=True)

NON_LABEL_COLS = {"id", "subreddit_source", "post_id", "w"}
HIGH_OVERLAP_THRESHOLD = 0.70  # flag pairs at or above this conditional match rate

## Load data and label categories

Label → source (`lloom`/`taxonomy`) comes directly from `agreement_results.csv` -- the same mapping already validated in 2.5.3, not rebuilt from scratch.

In [8]:
wide = pd.read_parquet(WIDE_PATH)
label_cols = [c for c in wide.columns if c not in NON_LABEL_COLS]

agreement = pd.read_csv(AGREEMENT_PATH)
label_sources = dict(zip(agreement["label"], agreement["source"]))

lloom_labels = [c for c in label_cols if label_sources.get(c) == "lloom"]
taxonomy_labels = [c for c in label_cols if label_sources.get(c) == "taxonomy"]

print(f"LLooM labels ({len(lloom_labels)}): {lloom_labels}")
print(f"Taxonomy labels ({len(taxonomy_labels)}): {taxonomy_labels}")

LLooM labels (8): ['Workplace Problem Guidance', 'Career Planning Advice', 'Emotional Support', 'Critical Pushback', 'Personal Relating', 'Discussion Direction', 'Practical Advice', 'Support Connections']
Taxonomy labels (6): ['informational_support', 'emotional_support', 'esteem_support', 'tangible_support', 'network_support', 'unsupportive_response']


## 1. Drop incomplete comments

Any comment missing even one of the 14 label values gets dropped entirely, so everything downstream works on a fully complete table -- no NaN handling needed in the co-occurrence math.

In [9]:
before = len(wide)
incomplete_mask = wide[label_cols].isna().any(axis=1)
n_incomplete = incomplete_mask.sum()

print(f"Comments with at least one missing label: {n_incomplete:,} / {before:,} ({n_incomplete/before:.3%})")

clean = wide[~incomplete_mask].copy()
print(f"Dropped {n_incomplete:,} comments -- {len(clean):,} remain, fully complete across all 14 labels.")

Comments with at least one missing label: 12 / 53,283 (0.023%)
Dropped 12 comments -- 53,271 remain, fully complete across all 14 labels.


## 2. Save `score_updated.jsonl` + `label_categories.json`

One JSON object per comment for the main file; label→category is a per-label fact, not per-comment, so it lives in a separate small metadata file rather than being repeated on every line.

In [10]:
out_path = os.path.join(OUTPUT_DIR, "score_updated.jsonl")
with open(out_path, "w") as f:
    for _, row in clean.iterrows():
        record = {
            "id": row["id"],
            "subreddit_source": row["subreddit_source"],
            "post_id": row["post_id"],
            "w": row["w"],
            "labels": {col: int(row[col]) for col in label_cols},
        }
        f.write(json.dumps(record) + "\n")
print(f"Saved: {out_path} ({len(clean):,} comments)")

meta_path = os.path.join(OUTPUT_DIR, "label_categories.json")
json.dump(
    [{"label": col, "source": label_sources.get(col, "unknown")} for col in label_cols],
    open(meta_path, "w"), indent=2
)
print(f"Saved: {meta_path}")

Saved: /Users/nadia/Desktop/redditRun_june/comment_data/score_updated.jsonl (53,271 comments)
Saved: /Users/nadia/Desktop/redditRun_june/comment_data/label_categories.json


## 2b. Label distribution -- how common is each of the 14 labels

Computed on `clean` (post-drop), so this matches exactly the same dataset the co-occurrence matrices below use -- not the pre-drop numbers from `score_stats.py`, which had a small amount of residual missingness.

`prevalence_weighted` uses the `w` column (corpus reweighting for sysadmin's subsampling) -- use this number, not `prevalence_unweighted`, for any claim about the real corpus rather than just this sample.

In [ ]:

# duplicate comments in each label presents this code block 
rows = []
for col in label_cols:
    n_matched = (clean[col] == 1).sum()
    print(f"{col}: {n_matched:,} comments")



Workplace Problem Guidance: 15,056 comments
Career Planning Advice: 21,346 comments
Emotional Support: 20,028 comments
Critical Pushback: 24,064 comments
Personal Relating: 27,119 comments
Discussion Direction: 24,415 comments
Practical Advice: 30,743 comments
Support Connections: 18,540 comments
informational_support: 41,492 comments
emotional_support: 16,835 comments
esteem_support: 14,524 comments
tangible_support: 11,194 comments
network_support: 12,763 comments
unsupportive_response: 10,233 comments


In [ ]:
# how many distinct comments have at least one label: 
# for each label, find comments where that column is 1 and every other label column sums to 0

for col in label_cols:
    only_this = clean[(clean[col] == 1) & (clean[label_cols].drop(columns=[col]).sum(axis=1) == 0)]
    print(f"{col} ONLY (no other label): {len(only_this):,}")

Workplace Problem Guidance ONLY (no other label): 5
Career Planning Advice ONLY (no other label): 19
Emotional Support ONLY (no other label): 22
Critical Pushback ONLY (no other label): 138
Personal Relating ONLY (no other label): 2,525
Discussion Direction ONLY (no other label): 441
Practical Advice ONLY (no other label): 7
Support Connections ONLY (no other label): 3
informational_support ONLY (no other label): 492
emotional_support ONLY (no other label): 9
esteem_support ONLY (no other label): 20
tangible_support ONLY (no other label): 4
network_support ONLY (no other label): 42
unsupportive_response ONLY (no other label): 137


In [27]:
only_one_counts = []
for col in label_cols:
    n = ((clean[col] == 1) & (clean[label_cols].drop(columns=[col]).sum(axis=1) == 0)).sum()
    only_one_counts.append({"label": col, "n_exclusive": n})

pd.DataFrame(only_one_counts).sort_values("n_exclusive", ascending=False)

,label,n_exclusive
4,Personal Relating,2525
8,informational_support,492
5,Discussion Direction,441
3,Critical Pushback,138
13,unsupportive_response,137
12,network_support,42
2,Emotional Support,22
10,esteem_support,20
1,Career Planning Advice,19
9,emotional_support,9


In [30]:
rows = []
for col in label_cols:
    total_col = (clean[col] == 1).sum()
    solely_col = ((clean[col] == 1) & (clean[label_cols].drop(columns=[col]).sum(axis=1) == 0)).sum()
    rate = solely_col / total_col if total_col > 0 else float("nan")
    rows.append({"label": col, "total": total_col, "solely": solely_col, "solo_rate": rate})

solo_rate_df = pd.DataFrame(rows).sort_values("solo_rate", ascending=False)
pd.set_option("display.float_format", lambda x: f"{x:.2%}" if 0 <= x <= 1 else f"{x:.0f}")
solo_rate_df

,label,total,solely,solo_rate
4,Personal Relating,27119,2525,9.31%
5,Discussion Direction,24415,441,1.81%
13,unsupportive_response,10233,137,1.34%
8,informational_support,41492,492,1.19%
3,Critical Pushback,24064,138,0.57%
12,network_support,12763,42,0.33%
10,esteem_support,14524,20,0.14%
2,Emotional Support,20028,22,0.11%
1,Career Planning Advice,21346,19,0.09%
9,emotional_support,16835,9,0.05%


In [28]:
label_distribution.to_csv(os.path.join(RESULTS_DIR, "label_distribution_clean.csv"), index=False)
print(f"Saved: {os.path.join(RESULTS_DIR, 'label_distribution_clean.csv')}")

Saved: /Users/nadia/Desktop/redditRun_june/comment_data/score_results_similarity/label_distribution_clean.csv


## Helper functions

`conditional_overlap_matrix`: `matrix[A][B] = P(B=1 | A=1)` -- of comments matching A, what % also match B. Not symmetric -- direction matters.

`flag_high_overlap`: scans a finished matrix and prints any cell at or above `HIGH_OVERLAP_THRESHOLD`.

In [13]:
def conditional_overlap_matrix(clean, row_labels, col_labels):
    """matrix[A][B] = P(B=1 | A=1) -- of comments matching A, what % also match B."""
    matrix = pd.DataFrame(index=row_labels, columns=col_labels, dtype=float)
    for a in row_labels:
        matched_a = clean[clean[a] == 1]
        n_a = len(matched_a)
        for b in col_labels:
            if n_a == 0:
                matrix.loc[a, b] = float("nan")
            else:
                matrix.loc[a, b] = (matched_a[b] == 1).mean()
    return matrix


def flag_high_overlap(matrix, row_labels, col_labels, same_group):
    print(f"Pairs at or above {HIGH_OVERLAP_THRESHOLD:.0%} conditional overlap:")
    found = False
    for a in row_labels:
        for b in col_labels:
            if same_group and a == b:
                continue
            val = matrix.loc[a, b]
            if pd.notna(val) and val >= HIGH_OVERLAP_THRESHOLD:
                print(f"  {val:.1%} of '{a}' comments also matched '{b}'")
                found = True
    if not found:
        print("  None found.")

## 3a. Within-LLooM co-occurrence (directional: row → column)

In [14]:
pd.set_option("display.float_format", lambda x: f"{x:.1%}" if pd.notna(x) else "  -  ")

lloom_matrix = conditional_overlap_matrix(clean, lloom_labels, lloom_labels)
lloom_matrix

,Workplace Problem Guidance,Career Planning Advice,Emotional Support,Critical Pushback,Personal Relating,Discussion Direction,Practical Advice,Support Connections
Workplace Problem Guidance,100.0%,43.7%,49.5%,53.8%,44.9%,62.0%,92.2%,52.3%
Career Planning Advice,30.8%,100.0%,51.8%,51.6%,46.0%,55.6%,80.7%,46.4%
Emotional Support,37.2%,55.3%,100.0%,41.4%,57.9%,45.9%,67.2%,37.6%
Critical Pushback,33.7%,45.7%,34.5%,100.0%,38.3%,64.7%,63.5%,34.1%
Personal Relating,24.9%,36.2%,42.7%,34.0%,100.0%,32.6%,50.1%,29.8%
Discussion Direction,38.2%,48.6%,37.6%,63.8%,36.2%,100.0%,72.3%,45.5%
Practical Advice,45.2%,56.0%,43.8%,49.7%,44.2%,57.4%,100.0%,56.9%
Support Connections,42.5%,53.5%,40.7%,44.2%,43.6%,59.9%,94.3%,100.0%


In [15]:
lloom_matrix.to_csv(os.path.join(RESULTS_DIR, "cooccurrence_within_lloom.csv"))
flag_high_overlap(lloom_matrix, lloom_labels, lloom_labels, same_group=True)

Pairs at or above 70% conditional overlap:
  92.2% of 'Workplace Problem Guidance' comments also matched 'Practical Advice'
  80.7% of 'Career Planning Advice' comments also matched 'Practical Advice'
  72.3% of 'Discussion Direction' comments also matched 'Practical Advice'
  94.3% of 'Support Connections' comments also matched 'Practical Advice'


## 3b. LLooM → taxonomy co-occurrence (directional: row → column)

In [16]:
cross_matrix = conditional_overlap_matrix(clean, lloom_labels, taxonomy_labels)
cross_matrix

,informational_support,emotional_support,esteem_support,tangible_support,network_support,unsupportive_response
Workplace Problem Guidance,96.4%,42.3%,38.4%,27.8%,23.2%,16.5%
Career Planning Advice,94.7%,44.6%,38.3%,22.6%,26.8%,18.1%
Emotional Support,84.4%,78.7%,59.8%,19.0%,41.5%,11.6%
Critical Pushback,87.2%,29.3%,28.0%,18.6%,18.1%,36.5%
Personal Relating,70.7%,35.4%,28.1%,18.3%,36.3%,12.2%
Discussion Direction,87.5%,32.3%,29.5%,25.7%,20.7%,27.4%
Practical Advice,98.0%,38.0%,32.6%,34.0%,23.1%,17.4%
Support Connections,97.5%,36.4%,31.0%,55.9%,26.2%,13.4%


In [17]:
cross_matrix.to_csv(os.path.join(RESULTS_DIR, "cooccurrence_lloom_vs_taxonomy.csv"))
flag_high_overlap(cross_matrix, lloom_labels, taxonomy_labels, same_group=False)

Pairs at or above 70% conditional overlap:
  96.4% of 'Workplace Problem Guidance' comments also matched 'informational_support'
  94.7% of 'Career Planning Advice' comments also matched 'informational_support'
  84.4% of 'Emotional Support' comments also matched 'informational_support'
  78.7% of 'Emotional Support' comments also matched 'emotional_support'
  87.2% of 'Critical Pushback' comments also matched 'informational_support'
  70.7% of 'Personal Relating' comments also matched 'informational_support'
  87.5% of 'Discussion Direction' comments also matched 'informational_support'
  98.0% of 'Practical Advice' comments also matched 'informational_support'
  97.5% of 'Support Connections' comments also matched 'informational_support'


## 3c. Taxonomy → LLooM (reverse direction, for completeness)

In [18]:
reverse_matrix = conditional_overlap_matrix(clean, taxonomy_labels, lloom_labels)
reverse_matrix

,Workplace Problem Guidance,Career Planning Advice,Emotional Support,Critical Pushback,Personal Relating,Discussion Direction,Practical Advice,Support Connections
informational_support,35.0%,48.7%,40.7%,50.6%,46.2%,51.5%,72.6%,43.6%
emotional_support,37.8%,56.5%,93.7%,41.8%,57.0%,46.8%,69.4%,40.1%
esteem_support,39.9%,56.3%,82.5%,46.4%,52.4%,49.6%,69.1%,39.5%
tangible_support,37.4%,43.1%,34.1%,39.9%,44.3%,56.1%,93.4%,92.6%
network_support,27.4%,44.9%,65.1%,34.2%,77.1%,39.6%,55.6%,38.1%
unsupportive_response,24.3%,37.9%,22.7%,85.9%,32.2%,65.5%,52.2%,24.3%


In [19]:
reverse_matrix.to_csv(os.path.join(RESULTS_DIR, "cooccurrence_taxonomy_vs_lloom.csv"))
flag_high_overlap(reverse_matrix, taxonomy_labels, lloom_labels, same_group=False)

Pairs at or above 70% conditional overlap:
  72.6% of 'informational_support' comments also matched 'Practical Advice'
  93.7% of 'emotional_support' comments also matched 'Emotional Support'
  82.5% of 'esteem_support' comments also matched 'Emotional Support'
  93.4% of 'tangible_support' comments also matched 'Practical Advice'
  92.6% of 'tangible_support' comments also matched 'Support Connections'
  77.1% of 'network_support' comments also matched 'Personal Relating'
  85.9% of 'unsupportive_response' comments also matched 'Critical Pushback'


## Interpretation reminder

A **high overlap in 3b** (e.g. most `Emotional Support` comments also matching `esteem_support`) suggests that LLooM concept may be largely redundant with the existing taxonomy label -- weaker evidence of a novel contribution.

**Low overlap across the board** supports the residual/novelty claim from 2.8.

The strongest signal for genuine redundancy is a pair that's **high in *both* directions** (3b and 3c together) -- a high number in only one direction usually just means one label is a broader category the other tends to fall inside of, not that the two are duplicates.